# Avito Cookie Bot Detection

В этом ноутбуке я сначала воспроизвожу базовую последовательность из quickstart: загружаю данные, осматриваю события, ограничиваю их observation window и строю простой baseline. Затем расширяю этот baseline признаками, которые описывают временную структуру и характер поведения cookie.

Цель — для каждой `cookie_id` из `test.csv` получить `score` от 0 до 1: чем выше score, тем выше вероятность автоматизированного трафика. Основной локальный критерий — официальный `Precision при Recall ≥ 70%` из `metric.py`.

Всё обучение выполняется локально на табличных данных. Для воспроизводимости фиксирую seed и версии библиотек.

In [ ]:
# Базовые импорты quickstart + фиксируем seed для воспроизводимости всего notebook.
import os
import re
import sys
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb

from metric import precision_at_recall, recall_at_fpr, pr_curve

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("data")
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("lightgbm:", lgb.__version__)
print("SEED:", SEED)

## 1. Загрузка данных

Начинаю с исходных файлов задачи. `target` находится только в train, а test используется только для построения признаков и получения финальных score. Метаданные окна нужны, чтобы для каждой cookie отделить доступную историю событий от событий за пределами observation window.

In [ ]:
train = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
test = pd.read_csv(
    DATA_DIR / "test.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
events = pd.read_csv(
    DATA_DIR / "events.csv.gz",
    parse_dates=["event_ts"],
)

print(train.shape, test.shape, events.shape)
print("доля ботов в train:", round(train.target.mean(), 4))
train.head()

In [ ]:
# Небольшая проверка схемы: одна строка train/test = одна cookie.
assert train["cookie_id"].is_unique
assert test["cookie_id"].is_unique
assert not set(train.cookie_id).intersection(test.cookie_id)

print("train columns:", train.columns.tolist())
print("test columns :", test.columns.tolist())
print("events columns:", events.columns.tolist())

In [ ]:
events.head()

## Осмотреться

Сначала повторяем базовый EDA из quickstart. Это не только описание данных: эти проверки помогают обнаружить пропуски, неоднородные значения и дубликаты до построения признаков.

In [ ]:
print(events.event_name.value_counts(dropna=False), "\n")
print(events.platform.value_counts(dropna=False), "\n")
print("пропуски по колонкам:")
print(events.isna().mean().round(3))

print("\nУникальных cookie_id в events:", events.cookie_id.nunique())
print("Дубликатов по (cookie_id, event_ts, event_name, item_id):",
      events.duplicated(["cookie_id", "event_ts", "event_name", "item_id"]).sum())
print("events отсортированы по event_ts:", events.event_ts.is_monotonic_increasing)

## Окно наблюдения

Как и в quickstart, события считаются только если

`window_start_ts <= event_ts < window_end_ts`.

Это принципиально: событие после `window_end_ts` недоступно модели в момент принятия решения и поэтому не должно попадать ни в один признак.

In [ ]:
def events_in_window(events_df, meta):
    """Оставляет только события внутри индивидуального окна каждой cookie.

    merge добавляет границы окна к каждому событию, после чего применяется
    строгий полуинтервал [start, end). validate защищает от случайного
    дублирования строк метаданных одной cookie.
    """
    assert meta["cookie_id"].is_unique
    ev = events_df.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )
    mask = ev["event_ts"].ge(ev["window_start_ts"]) & ev["event_ts"].lt(ev["window_end_ts"])
    return ev.loc[mask].copy()

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print("Событий в окне train:", len(ev_tr), "| test:", len(ev_te))

# Явно проверяем отсутствие look-ahead: после фильтра ни одного события за концом окна.
assert (ev_tr.event_ts < ev_tr.window_end_ts).all()
assert (ev_te.event_ts < ev_te.window_end_ts).all()
assert (ev_tr.event_ts >= ev_tr.window_start_ts).all()
assert (ev_te.event_ts >= ev_te.window_start_ts).all()

## Простейшие признаки — baseline

Сначала оставляю только два понятных агрегата: количество событий и число уникальных объявлений. Такой baseline позволяет понять, насколько много информации уже содержится в простом объёме активности.

In [ ]:
def basic_features(ev, meta):
    g = ev.groupby("cookie_id")
    f = pd.DataFrame({
        "n_events": g.size(),
        "item_nunique": g.item_id.nunique(),
    })
    f = meta[["cookie_id"]].merge(f.reset_index(), on="cookie_id", how="left")
    return f.fillna(0)

Xtr = basic_features(ev_tr, train)
Xte = basic_features(ev_te, test)
ytr = train.target.to_numpy()
Xtr.head()

## Валидация baseline

Для baseline использую временное разделение по `window_start_ts`. Это сохраняет порядок наблюдений во времени и даёт более реалистичную проверку переноса на более поздние окна, чем случайное перемешивание cookie.

In [ ]:
is_valid = train.window_start_ts.ge("2026-04-17").to_numpy()
cols = ["n_events", "item_nunique"]

baseline_rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=2,
)
baseline_rf.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid])
p_baseline = baseline_rf.predict_proba(Xtr.loc[is_valid, cols])[:, 1]

baseline_p = precision_at_recall(ytr[is_valid], p_baseline)
print("P@R0.7 baseline:", round(baseline_p, 4))
print("доля ботов на validation:", round(ytr[is_valid].mean(), 4))


# Расширение набора признаков

Простого количества событий недостаточно, поэтому дальше описываю поведение cookie с нескольких сторон:

1. объём и разнообразие активности;
2. временные интервалы, burst-поведение и последовательности событий;
3. User-Agent, поиск и cursor;
4. распределения категорий, seller type и платформ.

Все event-level признаки считаются только по событиям внутри индивидуального observation window.

## 2. Дубликаты

Сначала ограничиваю события observation window, а уже затем считаю дубликаты. Это не позволяет событиям за пределами окна повлиять на поведенческие признаки.

Повторяющиеся записи удаляю из behavioral stream, но количество повторов сохраняю отдельными признаками: сам факт повторов тоже может характеризовать поток.

In [ ]:
DUP_KEY = ["cookie_id", "event_ts", "event_name", "item_id"]

def prepare_event_stream(ev):
    """Возвращает очищенный behavioral stream и статистику дублей внутри окна."""
    ev = ev.copy()

    # Статистика строится до удаления дублей, но уже после фильтрации окна.
    dup = ev.groupby("cookie_id").agg(
        n_domain_duplicate_rows=("_domain_duplicate", "sum"),
        n_exact_duplicate_rows=("_exact_duplicate", "sum"),
    )

    # Для поведения оставляем одну запись каждого доменного duplicate-key.
    ev = ev.sort_values(DUP_KEY, kind="mergesort")
    ev = ev.drop_duplicates(DUP_KEY, keep="first").copy()
    return ev, dup

def mark_duplicates(ev):
    ev = ev.copy()
    ev["_domain_duplicate"] = ev.duplicated(DUP_KEY, keep=False).astype("int8")
    ev["_exact_duplicate"] = ev.duplicated(keep=False).astype("int8")
    return ev

ev_tr_raw = mark_duplicates(ev_tr)
ev_te_raw = mark_duplicates(ev_te)
ev_tr_clean, dup_tr = prepare_event_stream(ev_tr_raw)
ev_te_clean, dup_te = prepare_event_stream(ev_te_raw)

print("train events:", len(ev_tr), "->", len(ev_tr_clean))
print("test events :", len(ev_te), "->", len(ev_te_clean))

## 3. Временные и последовательностные признаки

Здесь описываю, как меняется активность во времени. После сортировки событий внутри каждой cookie межсобытийные интервалы и переходы между типами событий можно получить через `groupby().diff()` и `shift()`. Такой способ не требует вложенных циклов по событиям.

In [ ]:
def add_temporal_columns(ev):
    ev = ev.sort_values(["cookie_id", "event_ts"], kind="mergesort").copy()
    g = ev.groupby("cookie_id", sort=False)

    # Межсобытийные интервалы. Первый event имеет NaN — это корректно: у него нет предшественника.
    ev["_dt_prev"] = g["event_ts"].diff().dt.total_seconds()
    ev["_dt_next"] = -g["event_ts"].diff(-1).dt.total_seconds()

    # Время от первого события и время до конца разрешённого окна.
    first_ts = g["event_ts"].transform("min")
    ev["_elapsed_sec"] = (ev["event_ts"] - first_ts).dt.total_seconds()
    ev["_to_window_end_sec"] = (ev["window_end_ts"] - ev["event_ts"]).dt.total_seconds()

    # Переходы между событиями. START нужен, чтобы первое событие не потерялось.
    prev_name = g["event_name"].shift(1).astype("string").fillna("<START>")
    ev["_transition"] = prev_name + "->" + ev["event_name"].astype("string")

    # События в начале/конце окна — отдельные признаки концентрации активности.
    ev["_first_5m"] = ev["_elapsed_sec"].le(300).astype("int8")
    ev["_first_30m"] = ev["_elapsed_sec"].le(1800).astype("int8")
    ev["_last_5m"] = ev["_to_window_end_sec"].le(300).astype("int8")
    ev["_last_30m"] = ev["_to_window_end_sec"].le(1800).astype("int8")

    # Час UTC из timestamp. Мы не делаем вид, что знаем локальный часовой пояс пользователя.
    hour = ev["event_ts"].dt.hour
    ev["_night"] = hour.isin([0, 1, 2, 3, 4, 5]).astype("int8")
    ev["_hour_bin"] = (hour // 4).astype("int8")
    return ev

ev_tr_clean = add_temporal_columns(ev_tr_clean)
ev_te_clean = add_temporal_columns(ev_te_clean)

## 4. User-Agent, поиск и cursor

Сырой `user_agent` имеет высокую кардинальность, поэтому вместо него использую набор интерпретируемых regex-признаков: headless/bot/automation, Python HTTP-клиенты, CLI-инструменты и браузерные маркеры.

Для search учитываю частоту поиска, длину запросов и номера страниц. Для cursor — наличие координат, разброс и движение между последовательными событиями.

In [ ]:
UA_PATTERNS = {
    "ua_headless": r"headless",
    "ua_bot": r"\bbot\b|crawler|spider|slurp|scrapy|bingpreview|facebookexternalhit|semrush|ahrefs",
    "ua_python_http": r"python|requests|aiohttp|httpx|urllib|mechanicalsoup",
    "ua_cli": r"curl|wget|libwww|httpie",
    "ua_automation": r"phantomjs|selenium|playwright|puppeteer|webdriver",
    "ua_browser": r"chrome|chromium|firefox|safari|edg|opera",
    "ua_windows": r"windows",
    "ua_macos": r"mac os x|macintosh",
    "ua_linux": r"linux|x11",
    "ua_android": r"android",
    "ua_ios": r"iphone|ipad|ipod",
}

def add_behavior_columns(ev):
    ev = ev.copy()
    # Platform встречается в разных регистрах (WEB/Web/web, Android/android и т.п.).
    # Нормализация не использует target и только уменьшает искусственную кардинальность.
    ev["platform_norm"] = ev["platform"].fillna("<NA>").astype(str).str.strip().str.lower()
    ua = ev["user_agent"].fillna("").astype(str).str.lower()

    # Каждый regex превращается в бинарный флаг; долю таких событий затем считаем на cookie.
    for name, pattern in UA_PATTERNS.items():
        ev[name] = ua.str.contains(pattern, regex=True).astype("int8")
    ev["ua_len"] = ua.str.len().astype("int32")

    # Разделяем "был search" и "длина запроса": пустой search_query не должен считаться поиском.
    ev["has_search"] = ev["search_query"].notna().astype("int8")
    ev["query_nonempty"] = ev["search_query"].fillna("").astype(str).str.len().gt(0).astype("int8")
    ev["query_len"] = ev["search_query"].fillna("").astype(str).str.len()

    # Cursor motion.
    g = ev.groupby("cookie_id", sort=False)
    ev["dx"] = g["pointer_x"].diff()
    ev["dy"] = g["pointer_y"].diff()
    ev["move_dist"] = np.sqrt(ev["dx"].pow(2) + ev["dy"].pow(2))
    ev["move_speed"] = ev["move_dist"] / ev["_dt_prev"].replace(0, np.nan)
    # Ограничение только для агрегатов: исходные координаты не меняем.
    ev["move_speed_capped"] = ev["move_speed"].clip(upper=5000)
    return ev

ev_tr_clean = add_behavior_columns(ev_tr_clean)
ev_te_clean = add_behavior_columns(ev_te_clean)

## 5. Сборка признаков на уровне cookie

Теперь объединяю признаки в одну строку на `cookie_id`. Важное свойство builder-функций — они получают только события нужного набора и его metadata. `target` в feature engineering не используется.

Для категориальных полей считаю не только число уникальных значений и доли, но и концентрацию распределения через HHI и entropy.

In [ ]:
def distribution_features(ev, value_col, prefix):
    """HHI, entropy и доля доминирующего значения по cookie."""
    counts = ev.groupby(["cookie_id", value_col], dropna=False).size()
    totals = counts.groupby(level=0).sum()
    shares = counts / totals

    return pd.DataFrame({
        f"{prefix}_hhi": shares.pow(2).groupby(level=0).sum(),
        f"{prefix}_entropy": -(shares * np.log(shares.clip(lower=1e-12))).groupby(level=0).sum(),
        f"{prefix}_dominant_share": shares.groupby(level=0).max(),
    })

def build_features(ev, meta, dup_stats):
    """Строит только признаки, доступные к концу observation window."""
    if ev.empty:
        # Для пустого event stream возвращаем только meta; ниже общий код заполнит отсутствие событий.
        base = pd.DataFrame(index=pd.Index([], name="cookie_id"))
    else:
        g = ev.groupby("cookie_id", sort=False)

        # Базовые объёмные и diversity признаки.
        # n_days_active и platform_nunique считаем отдельно векторизованно,
        # чтобы не делать Python-lambda на каждой cookie.
        active_days = (
            ev.assign(_event_day=ev["event_ts"].dt.normalize())
              .groupby("cookie_id")["_event_day"].nunique()
        )
        base = g.agg(
            n_events=("event_name", "size"),
            n_event_types=("event_name", "nunique"),
            item_nunique=("item_id", "nunique"),
            category_nunique=("item_category", "nunique"),
            location_nunique=("item_location", "nunique"),
            seller_nunique=("seller_type", "nunique"),
            query_nunique=("search_query", "nunique"),
            platform_nunique=("platform_norm", "nunique"),
            ua_nunique=("user_agent", "nunique"),
            first_event_ts=("event_ts", "min"),
            last_event_ts=("event_ts", "max"),
        )

        base["n_days_active"] = active_days

        # Временная активность.
        span = (base["last_event_ts"] - base["first_event_ts"]).dt.total_seconds()
        base["active_span_sec"] = span
        base["active_hours"] = span / 3600.0

        for c in ["_dt_prev"]:
            base[f"{c}_mean"] = g[c].mean()
            base[f"{c}_std"] = g[c].std()
            base[f"{c}_min"] = g[c].min()
            base[f"{c}_max"] = g[c].max()
        base["dt_cv"] = base["_dt_prev_std"] / base["_dt_prev_mean"].replace(0, np.nan)

        # Burst rates: доля событий, пришедших вскоре после предыдущего.
        # Boolean mask + groupby.mean/sum — полностью векторизовано по событиям.
        for threshold in [1, 2, 5, 10, 30, 60]:
            mask = ev["_dt_prev"].le(threshold)
            base[f"frac_dt_le_{threshold}s"] = mask.groupby(ev["cookie_id"]).mean()
        base["n_bursts_dt_le_2s"] = ev["_dt_prev"].le(2).groupby(ev["cookie_id"]).sum()
        base["n_bursts_dt_le_10s"] = ev["_dt_prev"].le(10).groupby(ev["cookie_id"]).sum()

        # Концентрация событий в начале/конце окна.
        for c in ["_first_5m", "_first_30m", "_last_5m", "_last_30m", "_night", "has_search", "query_nonempty"]:
            base[c + "_frac"] = g[c].mean()

        # UA: средние бинарных флагов = доля событий с соответствующим паттерном.
        ua_cols = list(UA_PATTERNS.keys())
        ua_mean = g[ua_cols].mean().add_suffix("_frac")
        base = base.join(ua_mean)
        base["ua_len_mean"] = g["ua_len"].mean()
        base["ua_len_std"] = g["ua_len"].std()
        base["ua_len_max"] = g["ua_len"].max()

        # Доля доминирующего UA: бот может использовать один технический UA на всём потоке.
        ua_counts = ev.groupby(["cookie_id", "user_agent"], dropna=False).size()
        base["ua_dominant_share"] = ua_counts.groupby(level=0).max() / ua_counts.groupby(level=0).sum()

        # Search/query признаки. Query length считается только для непустых запросов.
        # Средняя длина только непустых запросов без groupby-apply:
        # сумма длин / число непустых запросов.
        nonempty = ev["query_len"].gt(0)
        query_sum = ev["query_len"].where(nonempty, 0).groupby(ev["cookie_id"]).sum()
        query_count = nonempty.groupby(ev["cookie_id"]).sum()
        base["query_len_mean_nonempty"] = query_sum / query_count.replace(0, np.nan)
        base["query_len_max"] = g["query_len"].max()
        base["search_page_mean"] = pd.to_numeric(ev["search_page"], errors="coerce").groupby(ev["cookie_id"]).mean()
        base["search_page_max"] = pd.to_numeric(ev["search_page"], errors="coerce").groupby(ev["cookie_id"]).max()

        # Cursor: доля событий с координатами и устойчивые агрегаты движения.
        base["cursor_frac"] = ev["pointer_x"].notna().groupby(ev["cookie_id"]).mean()
        base["cursor_x_std"] = g["pointer_x"].std()
        base["cursor_y_std"] = g["pointer_y"].std()
        base["move_dist_mean"] = g["move_dist"].mean()
        base["move_dist_std"] = g["move_dist"].std()
        base["move_dist_sum"] = g["move_dist"].sum()
        base["move_speed_median"] = g["move_speed_capped"].median()
        base["move_speed_p90"] = g["move_speed_capped"].quantile(0.90)

        # Производные rates.
        base["events_per_active_hour"] = base["n_events"] / base["active_hours"].replace(0, np.nan)
        base["events_per_item"] = base["n_events"] / base["item_nunique"].replace(0, np.nan)
        base["items_per_event"] = base["item_nunique"] / base["n_events"].replace(0, np.nan)
        base["events_per_category"] = base["n_events"] / base["category_nunique"].replace(0, np.nan)
        base["ua_per_event"] = base["ua_nunique"] / base["n_events"].replace(0, np.nan)

        # Event-name/platform/seller fractions. Они не требуют target.
        for value_col, prefix in [
            ("event_name", "event_frac"),
            ("platform_norm", "platform_frac"),
            ("seller_type", "seller_frac"),
        ]:
            ct = pd.crosstab(ev["cookie_id"], ev[value_col], normalize="index")
            ct.columns = [prefix + "__" + re.sub(r"[^0-9a-zA-Z_]+", "_", str(c)) for c in ct.columns]
            base = base.join(ct)

        # Распределения category/seller: HHI и entropy ловят концентрацию поведения.
        base = base.join(distribution_features(ev, "item_category", "category"))
        base = base.join(distribution_features(ev, "seller_type", "seller"))

        # Sequence без high-cardinality one-hot: число уникальных переходов,
        # dominant transition и entropy. Эти признаки не зависят от target.
        tr_counts = ev.groupby(["cookie_id", "_transition"]).size()
        tr_total = tr_counts.groupby(level=0).sum()
        tr_share = tr_counts / tr_total
        base["n_unique_transitions"] = tr_counts.groupby(level=0).size()
        base["transition_dominant_share"] = tr_share.groupby(level=0).max()
        base["transition_entropy"] = -(tr_share * np.log(tr_share.clip(lower=1e-12))).groupby(level=0).sum()

    # Meta-признаки допустимы: они известны до/к моменту окна.
    meta_idx = meta.set_index("cookie_id")
    base = meta_idx[["window_start_ts", "window_end_ts", "cookie_created_at"]].join(base, how="left")

    # Возраст cookie на старте окна — известен в момент наблюдения.
    base["cookie_age_at_start_sec"] = (
        base.index.to_series().map(meta_idx["window_start_ts"]) -
        base.index.to_series().map(meta_idx["cookie_created_at"])
    ).dt.total_seconds()

    # Время от начала окна до первого события — тоже не использует будущее за пределами окна.
    if "first_event_ts" in base:
        base["sec_to_first_event"] = (
            base["first_event_ts"] - base.index.to_series().map(meta_idx["window_start_ts"])
        ).dt.total_seconds()
        base["sec_from_first_to_last"] = (
            base["last_event_ts"] - base["first_event_ts"]
        ).dt.total_seconds()

    # Присоединяем duplicate statistics, которые были рассчитаны только внутри окна.
    base = base.join(dup_stats, how="left")

    # Убираем служебные timestamp columns: они не должны напрямую попадать в модель.
    base = base.drop(columns=[c for c in ["window_start_ts", "window_end_ts", "cookie_created_at", "first_event_ts", "last_event_ts"] if c in base], errors="ignore")

    # Важный семантический момент: отсутствие событий != NaN измеренного признака.
    base["has_events"] = base["n_events"].notna().astype("int8")
    return base

feat_tr = build_features(ev_tr_clean, train, dup_tr)
feat_te = build_features(ev_te_clean, test, dup_te)

print("Feature shapes:", feat_tr.shape, feat_te.shape)

## 6. Выравнивание train и test

После feature engineering у train и test должен быть один и тот же набор колонок. Редкие значения могут встретиться только в одном из наборов, поэтому колонки объединяю и затем выравниваю через `reindex`.

Для cookie без событий добавляю `has_events=0`. Счётчики и доли для таких cookie равны нулю, а статистики, которые невозможно вычислить без событий или интервалов, остаются `NaN`.

In [ ]:
# Полный список признаков — объединение колонок двух наборов; значения test не участвуют в обучении.
feature_columns = sorted(set(feat_tr.columns).union(feat_te.columns))
Xtr_full = feat_tr.reindex(columns=feature_columns).copy()
Xte_full = feat_te.reindex(columns=feature_columns).copy()

# Нельзя передавать timestamp/object в LightGBM как есть.
for df in [Xtr_full, Xte_full]:
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], errors="coerce")

# Счётчики и доли при отсутствии событий имеют естественный ноль.
zero_when_no_events = [
    c for c in Xtr_full.columns
    if c.startswith(("n_", "event_frac__", "platform_frac__", "seller_frac__"))
    or c.endswith("_nunique")
    or c.endswith("_frac")
]

for df in [Xtr_full, Xte_full]:
    no_events = df["has_events"].eq(0)
    for c in zero_when_no_events:
        if c in df.columns:
            df.loc[no_events, c] = 0.0
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

# cookie_id никогда не используется как feature.
assert "cookie_id" not in Xtr_full.columns
assert "target" not in Xtr_full.columns

Xtr_full = Xtr_full.astype(float)
Xte_full = Xte_full.astype(float)

print("Итоговое число признаков:", Xtr_full.shape[1])
print("Train cookies без событий:", int(Xtr_full.has_events.eq(0).sum()))
print("Test cookies без событий :", int(Xte_full.has_events.eq(0).sum()))

## 7. Контроль временной доступности признаков

Перед обучением отдельно проверяю, что очищенные event streams целиком лежат внутри `[window_start_ts, window_end_ts)`. Это гарантирует, что признаки не используют события, которые стали известны после окончания окна.

In [ ]:
# Повторная проверка временного окна на очищенных потоках.
for name, ev in [("train", ev_tr_clean), ("test", ev_te_clean)]:
    assert ev["event_ts"].ge(ev["window_start_ts"]).all()
    assert ev["event_ts"].lt(ev["window_end_ts"]).all()
    print(name, "window check: OK")

# Явно проверяем, что target не оказался в матрице признаков.
assert "target" not in Xtr_full.columns
assert "cookie_id" not in Xtr_full.columns

# Все score в дальнейшем будут получены только из модели, обученной на train.
print("Leakage checks: OK")

## 8. Temporal cross-validation

Для подбора модели использую несколько временных срезов. В каждом фолде train состоит из более ранних окон, а validation — из следующего временного блока. Validation-блоки между собой не пересекаются.

Такой split позволяет проверить, насколько выбранные признаки и модель переносятся на более поздние наблюдения.

In [ ]:
# Сортируем train по началу observation window.
train_order = np.argsort(train["window_start_ts"].to_numpy(), kind="mergesort")
n_train = len(train_order)

# Validation blocks: [50%,65%), [65%,80%), [80%,95%). Последние 5% оставляем
# как дополнительную временную зону, не используемую для выбора параметров.
fold_specs = [(0.50, 0.65), (0.65, 0.80), (0.80, 0.95)]
folds = []
for train_end, val_end in fold_specs:
    a = int(n_train * train_end)
    b = int(n_train * val_end)
    tr_idx = train_order[:a]
    va_idx = train_order[a:b]
    folds.append((tr_idx, va_idx))

for i, (tr_idx, va_idx) in enumerate(folds, 1):
    tr_dates = train.iloc[tr_idx].window_start_ts
    va_dates = train.iloc[va_idx].window_start_ts
    print(
        f"Fold {i}: train={len(tr_idx):,}, val={len(va_idx):,}; "
        f"train_end={tr_dates.max()}, val=[{va_dates.min()} .. {va_dates.max()}]"
    )

# Проверяем, что validation blocks не пересекаются.
for i in range(len(folds)):
    for j in range(i + 1, len(folds)):
        assert set(folds[i][1]).isdisjoint(set(folds[j][1]))
print("Validation blocks are disjoint: OK")

## 9. Метрика

Для выбора модели использую функции из официального `metric.py`. Главная метрика — `P@R≥0.70`: среди порогов, где recall не меньше 70%, выбирается максимальный precision.

PR-AUC и ROC-AUC считаю дополнительно, чтобы видеть общую картину качества ранжирования. Они не заменяют основную метрику.

In [ ]:
def score_predictions(y_true, score):
    return {
        "p_at_r_070": float(precision_at_recall(y_true, score, recall=0.70)),
        "pr_auc": float(average_precision_score(y_true, score)),
        "roc_auc": float(roc_auc_score(y_true, score)),
        "recall_at_fpr_001": float(recall_at_fpr(y_true, score, fpr=0.01)),
    }

# Self-check совпадает с примерами из официального metric.py.
assert precision_at_recall([1, 0, 1, 1, 1], [5, 4, 3, 2, 1]) == 0.8
assert precision_at_recall([1, 1, 0, 0], [0.5] * 4) == 0.5
assert precision_at_recall([0, 0, 1, 1], [0.5] * 4) == 0.5
assert recall_at_fpr(np.r_[np.ones(10), np.zeros(90)], np.full(100, 0.5)) == 0.0
print("Official metric self-check: OK")

# Базовая точка для полного feature set: константа 0.5.
print("Constant P@R>=0.70:", precision_at_recall(ytr, np.full(len(ytr), 0.5)))

## 10. Подбор LightGBM

Число деревьев выбираю непосредственно по основной метрике. Модель обучается до фиксированного максимума, после чего один и тот же booster оценивается на validation при разных `n_estimators`. Это позволяет не переобучать модель заново для каждой точки сетки.

Для нескольких конфигураций LightGBM сравниваю средний `P@R≥0.70` по temporal folds.

In [ ]:
ITER_GRID = [100, 200, 300, 500, 700, 900]
MAX_ESTIMATORS = max(ITER_GRID)

PARAM_GRID = [
    # Более глубокая модель: хорошо ловит нелинейные взаимодействия поведения.
    {
        "learning_rate": 0.03,
        "num_leaves": 63,
        "min_child_samples": 30,
        "colsample_bytree": 0.85,
        "subsample": 0.85,
        "subsample_freq": 1,
        "reg_alpha": 0.1,
        "reg_lambda": 0.3,
    },
    # Более регуляризованный вариант: проверка против переобучения.
    {
        "learning_rate": 0.02,
        "num_leaves": 31,
        "min_child_samples": 40,
        "colsample_bytree": 0.90,
        "subsample": 0.90,
        "subsample_freq": 1,
        "reg_alpha": 0.2,
        "reg_lambda": 0.5,
    },
]

cv_rows = []

for p_id, params in enumerate(PARAM_GRID):
    print(f"\nParameter set {p_id + 1}/{len(PARAM_GRID)}")
    for fold_id, (tr_idx, va_idx) in enumerate(folds, 1):
        # Один fit на fold; predict_proba(num_iteration=k) позволяет оценить
        # много вариантов числа деревьев без повторного обучения.
        model = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=MAX_ESTIMATORS,
            random_state=SEED,
            deterministic=True,
            force_col_wise=True,
            n_jobs=2,
            verbosity=-1,
            **params,
        )
        model.fit(Xtr_full.iloc[tr_idx], ytr[tr_idx])
        for n_trees in ITER_GRID:
            pred = model.predict_proba(
                Xtr_full.iloc[va_idx],
                num_iteration=n_trees,
            )[:, 1]
            metrics = score_predictions(ytr[va_idx], pred)
            cv_rows.append({
                "param_id": p_id,
                "fold": fold_id,
                "n_trees": n_trees,
                **metrics,
            })

cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(ARTIFACTS_DIR / "cv_results.csv", index=False)
print("CV rows:", len(cv_results))

### Выбор конфигурации

Сначала сравниваю конфигурации по среднему `P@R≥0.70`. Если значения близки, смотрю на стабильность между folds, а затем на PR-AUC и ROC-AUC как диагностические показатели.

In [ ]:
selection = (
    cv_results.groupby(["param_id", "n_trees"])
    .agg(
        p_at_r_070_mean=("p_at_r_070", "mean"),
        p_at_r_070_std=("p_at_r_070", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        roc_auc_mean=("roc_auc", "mean"),
        recall_at_fpr_001_mean=("recall_at_fpr_001", "mean"),
    )
    .reset_index()
    .sort_values(
        ["p_at_r_070_mean", "p_at_r_070_std", "pr_auc_mean", "roc_auc_mean"],
        ascending=[False, True, False, False],
    )
)

display(selection.head(15))

best_row = selection.iloc[0]
BEST_PARAM_ID = int(best_row.param_id)
BEST_N_TREES = int(best_row.n_trees)
BEST_PARAMS = PARAM_GRID[BEST_PARAM_ID].copy()

print("\nSelected parameter set:", BEST_PARAM_ID)
print("Selected n_trees:", BEST_N_TREES)
print("Mean CV P@R>=0.70:", round(best_row.p_at_r_070_mean, 4))
print("Params:", BEST_PARAMS)

## 11. Проверка стабильности выбранной конфигурации

После выбора конфигурации вывожу её результат отдельно на каждом временном fold. Это помогает увидеть, не получен ли высокий средний результат только за счёт одного периода.

## Tie diagnostics

Официальная метрика обрабатывает одинаковые score одной группой. Поэтому я не добавляю случайный шум к предсказаниям только ради искусственного разбиения одинаковых значений. Вместо этого проверяю, насколько часто ties встречаются на практике.

In [ ]:
best_cv_detail = cv_results[
    (cv_results.param_id == BEST_PARAM_ID) &
    (cv_results.n_trees == BEST_N_TREES)
].copy()
print(best_cv_detail[["fold", "p_at_r_070", "pr_auc", "roc_auc", "recall_at_fpr_001"]].to_string(index=False))

# Одинаковые score считаются одной группой официальной метрикой.
# Поэтому случайный шум к финальным предсказаниям не добавляется.
def tie_summary(scores):
    counts = pd.Series(scores).value_counts()
    return {
        "n_scores": len(scores),
        "n_unique": int(counts.size),
        "rows_in_ties": int(counts[counts > 1].sum()),
        "max_tie_size": int(counts.max()),
    }

# Для диагностики обучаем отдельную модель только на третьем fold.
tr3, va3 = folds[2]
diag_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=BEST_N_TREES,
    random_state=SEED,
    deterministic=True,
    force_col_wise=True,
    n_jobs=2,
    verbosity=-1,
    **BEST_PARAMS,
)
diag_model.fit(Xtr_full.iloc[tr3], ytr[tr3])
diag_pred = diag_model.predict_proba(
    Xtr_full.iloc[va3], num_iteration=BEST_N_TREES
)[:, 1]
print("Tie summary on selected validation fold:", tie_summary(diag_pred))
del diag_model, diag_pred


## 12. Финальная модель

После выбора гиперпараметров и числа деревьев переобучаю одну модель на всём train. Test в подборе не участвует: его score вычисляются только после того, как конфигурация уже зафиксирована.

In [ ]:
final_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=BEST_N_TREES,
    random_state=SEED,
    deterministic=True,
    force_col_wise=True,
    n_jobs=2,
    verbosity=-1,
    **BEST_PARAMS,
)
final_model.fit(Xtr_full, ytr)

p_test = final_model.predict_proba(Xte_full)[:, 1]

print("Test score summary:")
print(pd.Series(p_test).describe())
print("Tie diagnostics:", tie_summary(p_test))
assert np.isfinite(p_test).all()
assert np.all((p_test >= 0) & (p_test <= 1))

## 13. Submission

Финальный файл содержит только две колонки — `cookie_id` и `score` — и одну строку на каждую cookie из `test.csv`. Порядок строк сохраняю таким же, как в `test.csv`.

Итоговый файл сохраняется как `submission.csv` в корне проекта. Промежуточные таблицы CV и модель сохраняются в `artifacts/`.

In [ ]:
submission = pd.DataFrame({
    "cookie_id": test["cookie_id"].to_numpy(),
    "score": p_test,
})

# Сохраняю ровно один финальный submission в порядке test.csv.
assert submission.columns.tolist() == ["cookie_id", "score"]
assert len(submission) == len(test)
assert submission["cookie_id"].is_unique
assert submission["score"].notna().all()
assert np.isfinite(submission["score"]).all()
assert submission["score"].between(0, 1).all()

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head())


## 14. Воспроизводимость

Использую локальные open-source библиотеки `numpy`, `pandas`, `scikit-learn`, `lightgbm` и `joblib`. Seed фиксирован. Версии библиотек записываются в `requirements.txt`, а выбранная конфигурация модели — в `artifacts/run_config.json`.

In [ ]:
requirements_text = """# Python 3.14.7
numpy==2.3.5
pandas==2.2.3
scikit-learn==1.8.0
lightgbm==4.6.0
joblib==1.5.3
nbclient==0.10.4
nbformat==5.10.4
"""

Path("requirements.txt").write_text(requirements_text, encoding="utf-8")

import joblib
joblib.dump(
    {
        "model": final_model,
        "feature_columns": Xtr_full.columns.tolist(),
        "seed": SEED,
        "best_param_id": BEST_PARAM_ID,
        "best_n_trees": BEST_N_TREES,
        "best_params": BEST_PARAMS,
    },
    ARTIFACTS_DIR / "model_artifact.joblib",
)

run_config = {
    "python": sys.version,
    "seed": SEED,
    "best_param_id": BEST_PARAM_ID,
    "best_n_trees": BEST_N_TREES,
    "best_params": BEST_PARAMS,
    "n_features": int(Xtr_full.shape[1]),
}
Path(ARTIFACTS_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2, default=str), encoding="utf-8"
)
print("Reproducibility artifacts saved to", ARTIFACTS_DIR)


## Краткое описание подхода

**Признаки.** Для каждой cookie использую агрегаты объёма и разнообразия активности, межсобытийные интервалы и burst-признаки, распределение event types и платформ, переходы между событиями, User-Agent признаки, search/query, cursor motion, распределения category и seller type, HHI/entropy, возраст cookie и статистику дубликатов. Все event-признаки строятся только внутри observation window.

**Модель.** В качестве baseline использую Random Forest на `n_events` и `item_nunique`. Основная модель — LightGBM. Гиперпараметры и число деревьев выбираются по среднему официальному `P@R≥0.70` на нескольких temporal folds. PR-AUC и ROC-AUC используются как дополнительные диагностические метрики.

**Валидация.** Сортирую train по `window_start_ts` и использую expanding temporal split с непересекающимися validation-блоками. После выбора конфигурации модель обучается на всём train и применяется к test.